In [ ]:
import jieba
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
import nltk
from nltk.corpus import stopwords

# 載入停用詞
首先，載入NLTK內建的中文停用詞為初始資料

In [ ]:
nltk.download('stopwords')
stopWord = set(stopwords.words('chinese'))

## 僅為示範綜合效果，繼續疊加中文停用詞表、哈爾濱工業大學停用詞表、百度停用詞表、四川大學機器智能實驗室停用詞表

In [ ]:
with open('./data/hit_stopwords.txt','r',encoding='utf-8') as f:   #載入哈爾濱工業大學停用詞
    for word in f.readlines():
        stopWord.update(word.strip())
with open('./data/cn_stopwords.txt','r',encoding='utf-8') as f:   #載入簡體中文停用詞
    for word in f.readlines():
        stopWord.update(word.strip())
with open('./data/baidu_stopwords.txt','r',encoding='utf-8') as f:   #載入百度停用詞
    for word in f.readlines():
        stopWord.update(word.strip())
with open('./data/scu_stopwords.txt','r',encoding='utf-8') as f:   #載入四川大學機器智能實驗室停用詞
    for word in f.readlines():
        stopWord.update(word.strip())

# 載入電影評論與其標籤，進行斷詞、去除停用詞

In [ ]:
import re
def cutword(line):
    review=re.sub(r'[a-zA-Z0-9]*','',line)
    wordList=jieba.lcut(review,cut_all=False)
    return ' '.join([word for word in wordList if word not in stopWord and len(word)>1])
df_zh = pd.read_csv('./data/movie_data_zh.csv', encoding='utf-8')
df_zh['word_list']=df_zh['review'].apply(cutword)
X = df_zh['word_list'].to_numpy()
y = df_zh['sentiment'].to_numpy()
df_zh.head(10)

#### 為方便未來再使用前面處理的結果，將斷詞處理完成的dataframe資料壓縮存檔。

In [ ]:
df_zh.to_csv("./data/movie_data_zh_jieba.csv.xz",compression='xz',index=False)

#### 下次使用之前可省略前面的斷詞程序，直接從檔案還原為dataframe 物件

In [ ]:
df_zh=pd.read_csv('./data/movie_data_zh_jieba.csv.xz',compression='xz')

# 給予可能的各參數如下，經過交叉驗證尋找超參數最佳組合
**TfidfVectorizer的選項：vect__超參數名稱**

**LogisticRegression的選項：clf__超參數名稱**

      ngram_range=(1,2) 加入雙詞特徵，
      max_features 限制維度
      max_df=0.8:去除出現在80%文檔中的詞
      min_df=2:只保留至少出現在2個文本中的詞
    總共需要擬合驗證數: (fits)     
      超參數組合數: 2x2x2x2x2x3x2x2 + 1x1x1x2x2x2x2x2x2=384 + 64 = 448
      擬合5個folds: 448x5 = 2240
      超參數組合衝突: 2x2x2x2x2x1x2x1 = 64x5 = 320
      實際擬合驗證數: 2240 - 320 = 1920
X part 已完成斷詞，因此超參數`tokenizer`仍舊維持None。

**須注意運行時間的取捨**

      運行下列程式碼可能超過200分鐘，你可以考慮縮小資料規模(包括訓練樣本或甚至超參數`max_features`)節省時間，但可能得出效果較弱的模型，或者，減少個別超參數的元素來降低超參數的組合數，例如下列`param_grid`先測其中一個，去除不必要的參數元素後，再測另外一個。

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

tfidf = TfidfVectorizer(token_pattern=r'(?u)\b\w\w+\b',
                        max_df=0.8,min_df=2)
# Pipeline of tfidf+logisticregression
tfidf_lreg_pipeline = Pipeline([      
    ('vect', tfidf), 
    ('clf', LogisticRegression(random_state=0))])
param_grid = [{'vect__ngram_range': [(1,1),(1,2)],
               'vect__norm':['l1','l2'],
               'vect__max_features':[1000,1500],
               'vect__max_df':[0.8,0.9],
               'vect__min_df':[2,4],
               'clf__l1_ratio': [0,1,0.5],
               'clf__C': [1.0, 10.0],
               'clf__solver': ['liblinear','saga']},
              {'vect__ngram_range': [(1,1)],
               'vect__use_idf':[False],
               'vect__norm':[None],
               'vect__max_features':[1000,1500],
               'vect__max_df':[0.7,0.8],
               'vect__min_df':[2,6],
               'clf__l1_ratio': [0,1],
               'clf__C': [1.0, 10.0],
               'clf__solver': ['liblinear','lbfgs']},
              ]
gs_lr_tfidf = GridSearchCV(tfidf_lreg_pipeline, param_grid,     # 評估
                           scoring='accuracy',
                           cv=5,
                           verbose=2,
                           n_jobs=-1)
gs_lr_tfidf.fit(X, y)

上面程式分成5疊(5 fold)樣本進行交叉驗證正確率(accuracy)，其最後得出最佳準確度(`best_score_`)係為此5次的平均。

In [ ]:
print('最佳超參數集: %s ' % gs_lr_tfidf.best_params_)
print('交叉驗證最佳準確度: %.3f' % gs_lr_tfidf.best_score_)

# 擬合、轉換
#### 使用如上交叉驗證得出的最佳超參數組：{'clf__C': 1.0, 'clf__l1_ratio': 0.5, 'clf__solver': 'saga', 'vect__max_df': 0.8, 'vect__max_features': 1500, 'vect__min_df': 2, 'vect__ngram_range': (1, 1), 'vect__norm': 'l2'} 

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.01, random_state=42)
vectorizer = TfidfVectorizer(ngram_range=(1,1), max_features=1500,
                             norm='l2',
                             max_df=0.8,min_df=2)
X_train_tfidf = vectorizer.fit_transform(X_train)
model = LogisticRegression(C=1.0, l1_ratio=0.5,solver='saga')  # C 為正則化強度
model.fit(X_train_tfidf, y_train)

In [ ]:
print("數字向量詞彙集:",vectorizer.vocabulary_.items())
y_pred_train = model.predict(X_train_tfidf)
print("訓練集準確率:",accuracy_score(y_train, y_pred_train))
X_test_tfidf = vectorizer.transform(X_test)
y_pred_test = model.predict(X_test_tfidf)
print("測試集準確率:",accuracy_score(y_test, y_pred_test))
print(classification_report(y_test, y_pred_test))
y_pred_proba=model.predict_proba(X_test_tfidf)
df_1 = pd.DataFrame({'真實標籤':y_test,'預測標籤':y_pred_test,'類別0機率':y_pred_proba[:,0],'類別1機率':y_pred_proba[:,1]})
df_1[(df_1['預測標籤']!=df_1['真實標籤'])]

#### 繼續細分saga下features數與elastic net 係數觀其是否能有更好的效果

In [ ]:
param_grid = [
              {'clf__l1_ratio': [0.3,0.5,0.8],
               'clf__C': [1.0],
               'clf__solver': ['saga'],
               'vect__max_df':[0.8],
               'vect__max_features':[1500,5000,10000,20000],
               'vect__min_df':[2],
               'vect__ngram_range': [(1,1)],
               'vect__norm':['l1','l2',None]}           
              ]
gs_lr_tfidf = GridSearchCV(tfidf_lreg_pipeline, param_grid,     # 評估
                           scoring='accuracy',
                           cv=5,
                           verbose=2,
                           n_jobs=-1)
gs_lr_tfidf.fit(X, y)

In [ ]:
print('最佳超參數集: %s ' % gs_lr_tfidf.best_params_)
print('交叉驗證最佳準確度: %.3f' % gs_lr_tfidf.best_score_)

#### 最佳超參數集如上'clf__l1_ratio': 0.3,'vect__max_features': 20000可獲得更好的學習效果
#### max_features=20000是上述交叉驗證給予[1500,5000,10000,20000]的最大值，若繼續加大max_features亦有可能提升準確度，但學習資源的使用亦會更昂貴，兩者需要尋求平衡

In [ ]:
X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(X, y, test_size=0.01, random_state=42)
vectorizer_2 = TfidfVectorizer(ngram_range=(1,1), max_features=20000,
                             norm=None,
                             max_df=0.8,min_df=2)
X_train_tfidf_2 = vectorizer_2.fit_transform(X_train_2)
model_2 = LogisticRegression(C=1.0, l1_ratio=0.3,solver='saga')  # C 為正則化強度
model_2.fit(X_train_tfidf_2, y_train_2)

In [ ]:
print("數字向量詞彙集:",vectorizer_2.vocabulary_.items())
y_pred_train_2 = model_2.predict(X_train_tfidf_2)
print("訓練集準確率:",accuracy_score(y_train_2, y_pred_train_2))
X_test_tfidf_2 = vectorizer_2.transform(X_test_2)
y_pred_test_2 = model_2.predict(X_test_tfidf_2)
print("測試集準確率:",accuracy_score(y_test_2, y_pred_test_2))
print(classification_report(y_test_2, y_pred_test_2))
y_pred_proba_2=model_2.predict_proba(X_test_tfidf_2)
df_2 = pd.DataFrame({'真實標籤':y_test_2,'預測標籤':y_pred_test_2,'類別0機率':y_pred_proba_2[:,0],'類別1機率':y_pred_proba_2[:,1]})

In [ ]:
df_2[(df_2['預測標籤']!=df_2['真實標籤'])]

In [ ]:
param_grid = [
              {'clf__l1_ratio': [0.3,0.5,0.8],
               'clf__C': [1.0],
               'clf__solver': ['saga'],
               'vect__max_df':[0.8],
               'vect__max_features':[20000,30000],
               'vect__min_df':[2],
               'vect__ngram_range': [(1,1)],
               'vect__norm':['l1','l2',None]},
               {'clf__l1_ratio': [0],
               'clf__C': [1.0],
               'clf__solver': ['sag'],
               'vect__max_df':[0.8],
               'vect__max_features':[10000,20000,30000],
               'vect__min_df':[2],
               'vect__ngram_range': [(1,1)],
               'vect__norm':['l1','l2',None]}         
              ]
gs_lr_tfidf = GridSearchCV(tfidf_lreg_pipeline, param_grid,     # 評估
                           scoring='accuracy',
                           cv=5,
                           verbose=2,
                           n_jobs=-1)
gs_lr_tfidf.fit(X, y)

加大max_features至30000提升準確度並不顯著，但學習資源的使用更昂貴，兩者需要尋求平衡

In [ ]:
print('最佳超參數集: %s ' % gs_lr_tfidf.best_params_)
print('交叉驗證最佳準確度: %.3f' % gs_lr_tfidf.best_score_)

In [ ]:
X_train_3, X_test_3, y_train_3, y_test_3 = train_test_split(X, y, test_size=0.01, random_state=42)
vectorizer_3 = TfidfVectorizer(ngram_range=(1,1), max_features=30000,
                             norm=None,
                             max_df=0.8,min_df=2)
X_train_tfidf_3 = vectorizer_3.fit_transform(X_train_3)
model_3 = LogisticRegression(C=1.0, l1_ratio=0.3,solver='saga')  # C 為正則化強度
model_3.fit(X_train_tfidf_3, y_train_3)

In [ ]:
y_pred_train_3 = model_3.predict(X_train_tfidf_3)
print("訓練集準確率:",accuracy_score(y_train_3, y_pred_train_3))
X_test_tfidf_3 = vectorizer_3.transform(X_test_3)
y_pred_test_3 = model_3.predict(X_test_tfidf_3)
print("測試集準確率:",accuracy_score(y_test_3, y_pred_test_3))